# Baseline-Referenced Wiener Spectral Gating for DBS Artifact Removal

**Problem**: Standard notch filters remove ALL power at DBS harmonics (7, 14, 21, ... Hz), destroying endogenous theta, alpha, and beta activity at those frequencies.

**Solution**: Use clean baseline recordings (XUAWAKEPRE / XUSLEEP) to determine how much power *should* exist at each frequency. Only remove the *excess* power introduced by DBS stimulation.

**Method**: Wiener spectral gating — at each harmonic, compute `G(f) = P_baseline(f) / P_dbs(f)`. This gain is ~1 away from harmonics (untouched) and < 1 only where DBS adds excess power.

In [ ]:
import sys
import os
import numpy as np
import mne
import matplotlib.pyplot as plt
from scipy import signal

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.filters import BaselineReferencedFilter
from src.preprocessing import EEGPreprocessor

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100

## 1. Load and Standardize Data

In [ ]:
# Initialize preprocessor for channel standardization
preprocessor = EEGPreprocessor(input_dir="data/XU/", output_dir="data/processed/")

data_dir = os.path.join(project_root, "data", "XU")

# Load clean baseline (no DBS) — match state to your DBS recording
raw_baseline = mne.io.read_raw_edf(
    os.path.join(data_dir, "XUAWAKEPRE_deidentified.edf"), preload=True, verbose='WARNING'
)
raw_baseline = preprocessor.standardize_channels(raw_baseline)
raw_baseline.filter(l_freq=1.0, h_freq=70.0, fir_design='firwin', phase='zero', verbose='WARNING')

# Load DBS-contaminated recording (7 Hz)
raw_dbs = mne.io.read_raw_edf(
    os.path.join(data_dir, "XUAWAKE7_deidentified.edf"), preload=True, verbose='WARNING'
)
raw_dbs = preprocessor.standardize_channels(raw_dbs)
raw_dbs.filter(l_freq=1.0, h_freq=70.0, fir_design='firwin', phase='zero', verbose='WARNING')

print(f"Baseline: {raw_baseline.info['nchan']} channels, {raw_baseline.n_times} samples, {raw_baseline.info['sfreq']} Hz")
print(f"DBS:      {raw_dbs.info['nchan']} channels, {raw_dbs.n_times} samples, {raw_dbs.info['sfreq']} Hz")

## 2. Apply Baseline-Referenced Wiener Filter vs. Old Notch Filter

In [ ]:
# --- Method A: NEW Baseline-Referenced Wiener Filter ---
wiener_filter = BaselineReferencedFilter(
    baseline_raw=raw_baseline,
    dbs_freq=7.0,
    harmonic_bandwidth=1.5,  # Only touches +/- 0.75 Hz around each harmonic
    n_fft=4096,              # Fine frequency resolution (~0.06 Hz at 256 Hz sfreq)
    floor_db=-40.0           # Never fully null — retain -40 dB floor
)
raw_wiener = wiener_filter.filter(raw_dbs)

# --- Method B: OLD Notch Filter (current pipeline) ---
raw_notch = raw_dbs.copy()
sfreq = raw_notch.info['sfreq']
nyquist = sfreq / 2.0
dbs_harmonics = np.arange(7.0, nyquist, 7.0)
# Deduplicate close freqs (same logic as current remove_artifacts)
notch_widths = 2.0
min_sep = notch_widths + 2.0
deduped = []
for f in sorted(dbs_harmonics):
    if not deduped or (f - deduped[-1]) >= min_sep:
        deduped.append(f)
raw_notch.notch_filter(freqs=deduped, fir_design='firwin', phase='zero',
                       notch_widths=notch_widths, verbose='WARNING')

print("Both methods applied.")

## 3. PSD Comparison: Baseline vs. DBS Raw vs. Wiener vs. Notch

The key diagnostic: at 7, 14, 21 Hz etc., the Wiener curve should match the baseline level (brain activity preserved), while the notch curve shows deep valleys (brain activity destroyed).

In [ ]:
def compute_avg_psd(raw, n_fft=4096):
    """Compute channel-averaged PSD in dB/Hz."""
    data = raw.get_data()
    freqs, psd = signal.welch(data, fs=raw.info['sfreq'], nperseg=n_fft,
                              noverlap=n_fft // 2, window='hann', axis=1)
    psd_db = 10 * np.log10(psd + 1e-30)
    return freqs, psd_db.mean(axis=0)  # Average across channels

freqs_b, psd_baseline = compute_avg_psd(raw_baseline)
freqs_d, psd_dbs_raw  = compute_avg_psd(raw_dbs)
freqs_w, psd_wiener   = compute_avg_psd(raw_wiener)
freqs_n, psd_notch    = compute_avg_psd(raw_notch)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Full spectrum view
ax = axes[0]
ax.plot(freqs_b, psd_baseline, 'k-', alpha=0.7, linewidth=2, label='Baseline (no DBS)')
ax.plot(freqs_d, psd_dbs_raw, 'r-', alpha=0.4, linewidth=1, label='DBS Raw (7 Hz)')
ax.plot(freqs_w, psd_wiener, 'b-', alpha=0.8, linewidth=1.5, label='Wiener (NEW)')
ax.plot(freqs_n, psd_notch, 'g--', alpha=0.6, linewidth=1.5, label='Notch (OLD)')
ax.set_xlim(1, 70)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Power (dB/Hz)')
ax.set_title('Full Spectrum PSD Comparison')
ax.legend()
ax.grid(True, alpha=0.3)

# Zoom into theta-alpha range (4-30 Hz) where damage is most visible
ax = axes[1]
ax.plot(freqs_b, psd_baseline, 'k-', alpha=0.7, linewidth=2, label='Baseline (no DBS)')
ax.plot(freqs_d, psd_dbs_raw, 'r-', alpha=0.4, linewidth=1, label='DBS Raw (7 Hz)')
ax.plot(freqs_w, psd_wiener, 'b-', alpha=0.8, linewidth=1.5, label='Wiener (NEW)')
ax.plot(freqs_n, psd_notch, 'g--', alpha=0.6, linewidth=1.5, label='Notch (OLD)')
ax.set_xlim(4, 35)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Power (dB/Hz)')
ax.set_title('Zoomed: Theta-Alpha-Beta Range (4-35 Hz)')
ax.legend()
ax.grid(True, alpha=0.3)

# Mark DBS harmonics
for ax in axes:
    for h in np.arange(7, 70, 7):
        ax.axvline(h, color='red', alpha=0.15, linestyle=':', linewidth=0.8)

plt.tight_layout()
plt.savefig(os.path.join(project_root, 'figures', 'wiener_vs_notch_psd_comparison.png'),
            dpi=150, bbox_inches='tight')
plt.show()

## 4. Quantitative Validation: Band Power Preservation

Compare band power (delta, theta, alpha, beta) between baseline and each filtered output. The Wiener method should preserve band power close to baseline levels.

In [ ]:
bands = {
    'Delta (1-4 Hz)':  (1, 4),
    'Theta (4-8 Hz)':  (4, 8),
    'Alpha (8-13 Hz)': (8, 13),
    'Beta (13-30 Hz)': (13, 30),
    'Gamma (30-70 Hz)': (30, 70),
}

def band_power(freqs, psd_linear, fmin, fmax):
    """Integrate PSD over a frequency band (trapezoidal rule)."""
    mask = (freqs >= fmin) & (freqs <= fmax)
    return np.trapz(psd_linear[mask], freqs[mask])

# Recompute PSDs in linear scale for proper integration
def compute_avg_psd_linear(raw, n_fft=4096):
    data = raw.get_data()
    freqs, psd = signal.welch(data, fs=raw.info['sfreq'], nperseg=n_fft,
                              noverlap=n_fft // 2, window='hann', axis=1)
    return freqs, psd.mean(axis=0)

f_b, psd_b_lin = compute_avg_psd_linear(raw_baseline)
f_d, psd_d_lin = compute_avg_psd_linear(raw_dbs)
f_w, psd_w_lin = compute_avg_psd_linear(raw_wiener)
f_n, psd_n_lin = compute_avg_psd_linear(raw_notch)

print(f"{'Band':<20} {'Baseline':>12} {'DBS Raw':>12} {'Wiener':>12} {'Notch':>12}  {'Wiener err%':>12} {'Notch err%':>12}")
print("-" * 100)

for name, (fmin, fmax) in bands.items():
    bp_base = band_power(f_b, psd_b_lin, fmin, fmax)
    bp_dbs  = band_power(f_d, psd_d_lin, fmin, fmax)
    bp_wien = band_power(f_w, psd_w_lin, fmin, fmax)
    bp_notc = band_power(f_n, psd_n_lin, fmin, fmax)

    err_wien = 100 * (bp_wien - bp_base) / bp_base
    err_notc = 100 * (bp_notc - bp_base) / bp_base

    print(f"{name:<20} {bp_base:12.4e} {bp_dbs:12.4e} {bp_wien:12.4e} {bp_notc:12.4e}  {err_wien:+11.1f}% {err_notc:+11.1f}%")

## 5. Integration with Pipeline

To use the new filter in your existing `EEGPreprocessor` pipeline, replace `remove_artifacts()` with `remove_artifacts_baseline()`:

```python
# In your pipeline, replace step 3:
# OLD: raw = preprocessor.remove_artifacts(raw)
# NEW:
raw = preprocessor.remove_artifacts_baseline(
    raw, 
    baseline_raw=raw_baseline,  # Your clean XUAWAKEPRE or XUSLEEP
    dbs_freq=7.0,
    harmonic_bandwidth=1.5,
    n_fft=4096
)
```

### Tunable Parameters
| Parameter | Default | What it controls |
|---|---|---|
| `harmonic_bandwidth` | 1.5 Hz | Width of the spectral gate around each harmonic. Narrower = safer but may miss wide artifacts. Wider = more aggressive. |
| `n_fft` | 4096 | Frequency resolution. Higher = finer separation of DBS peak from brain. At 256 Hz sfreq, 4096 gives ~0.06 Hz resolution. |
| `floor_db` | -40 dB | Minimum gain — prevents complete nulling. -40 dB = 99.99% removal. Set to -20 dB for more conservative filtering. |